            # Mexicali Urban Liveability Index
            ## Overview, output schema and a worked example

            **Read this notebook before starting your own work
            package.** It explains what is being built, what you are
            expected to deliver, and shows one indicator carried all
            the way from a row in the workbook to a validated,
            ingestible deliverable.

            Schema version **1.0.0**.

            ---

            ### What is being built

            A suite of spatial liveability indicators for Mexicali, an
            arid city in Baja California, and from them a composite
            liveability index. Outputs also feed the **Reimagina
            Urbana** platform.

            The local team identified a large set of candidate
            indicators from a review of the liveability literature,
            classified them by domain, subdomain, category and subject,
            and narrowed them to **81 indicators** that are
            both plausibly relevant to health and wellbeing in Mexicali
            and feasible with available data.

            Indicators are *adapted*, not copied, from the reviewed
            articles. That is precisely why each one needs independent
            health evidence: the source article establishes that the
            indicator has been used, not that it matters.

            ### Work packages

            | Code | Work package | Lead | Indicators |
            |---|---|---|---|
            | `WP00_composite_index` | Composite liveability index | Carl Higgs | 1 |
| `WP01_ghsci_access_network` | Destination access and network measures (GHSCI) | Carl Higgs | 39 |
| `WP02_thermal_comfort_and_heat` | Thermal comfort and urban heat | Rossano Schifanella | 7 |
| `WP03_air_quality` | Air quality | TBC | 6 |
| `WP04_greenness_and_land_cover` | Greenness and land cover | TBC | 5 |
| `WP05_hazards_and_incidents` | Environmental hazards and safety incidents | TBC | 8 |
| `WP06_urban_form_and_land_use` | Urban form and land use | TBC | 7 |
| `WP07_street_infrastructure` | Street and active travel infrastructure | TBC | 4 |
| `WP08_housing_economy_and_services` | Housing, economy and public services | TBC | 4 |

            ### Analytical lenses (*enfoques*)

            One indicator can be measured several ways. The team
            defined these lenses:

            | Lens | Meaning |
            |---|---|
            | `proximity` | Distance to the closest instance (m), measured along the pedestrian network unless otherwise documented |
| `accessibility` | Whether an instance is reachable within a policy-relevant threshold; reported as the share of population (or of the unit) meeting the threshold |
| `quantity` | Count or amount within an area or threshold distance |
| `density` | Amount relative to another unit (per km², per 1,000 persons, or percentage coverage) |
| `diversity` | Mix or evenness (e.g. entropy) within an area |
| `quality` | A graded or composite assessment of condition or suitability |
| `equity` | Distributional summary across the population. Derived centrally from the finest-scale results -- analysts do not compute this lens themselves (see the schema document) |

            ### Reporting geographies

            | Level | Description | Required |
            |---|---|---|
            | `grid_100m` | 100 m regular grid over the study region (optional; supply only where the source data are genuinely this fine) | optional |
| `manzana` | INEGI census block (manzana); geo_id is the 16-character CVEGEO | required |
| `condesa_lote` | Individual lot within the Condesa new development area (optional detail scale) | optional |
| `condesa_fraccionamiento` | Condesa new-development subdivision polygon; the project focus area in south-east Mexicali | required |
| `grid_1000m` | 1 km regular grid over the study region | required |
| `ageb` | INEGI basic geostatistical area (AGEB); geo_id is the 13-character CVEGEO | required |
| `city` | The Mexicali study region as a whole (single unit) | required |

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

In [ ]:
register = uli.register.load()
register.groupby('work_package').agg(
    indicators=('indicator_id', 'size'),
    composites=('is_composite', 'sum'),
)

## The reference geographies

Everyone reports against the same units, read from one
geopackage in EPSG:6366. Do not build your own grid.

In [ ]:
summary = pd.DataFrame([
    {
        'geo_level': level,
        'units': len(uli.geography.units(level)),
        'median_area_ha': round(
            uli.geography.units(level)['area_sqm'].median()
            / 10000, 2),
        'population_ghs_pop_2025': round(
            uli.geography.units(level)['population'].sum()),
        'population_census_2020': round(
            uli.geography.units(level)[
                'pop_census_2020'].sum()),
    }
    for level in uli.vocab.GEO_RESOLUTION_ORDER
])
summary

In [ ]:
# Where is Condesa?
fig, ax = plt.subplots(figsize=(11, 6))
uli.geography.load('city').boundary.plot(
    ax=ax, color='0.6', linewidth=0.8)
uli.geography.load('manzana').plot(
    ax=ax, color='0.85', edgecolor='none')
uli.geography.load('condesa_fraccionamiento').plot(
    ax=ax, color='crimson', edgecolor='crimson')
ax.set_title('Mexicali ULI study extent, with the Condesa '
             'new development in red')
ax.set_axis_off()

---
## Worked example

An indicator carried end to end, using data already in the
repository: **access to convenience destinations**, from
the GHSCI Mexicali destinations layer.

The example is deliberately small. What matters is the
shape: documentation first, then a native-scale
calculation, then harmonisation, validation and delivery.

In [ ]:
GHSCI = os.path.join(
    '..', '..', '..', '_study_region_outputs',
    'MX_Mexicali_2025-MZA',
    'MX_Mexicali_2025-MZA_1600m_buffer.gpkg')

destinations = gpd.read_file(GHSCI, layer='destinations')
destinations['dest_name'].value_counts().head(10)

In [ ]:
# 1. Documentation.  Every field below is a real
#    requirement, not an example of one.
example = uli.metadata_stub(132, analyst=ANALYST)   # minimarts

example['indicator']['status'] = 'draft'
example['rationale']['statement'] = (
    'Small food retail within walking distance supports '
    'walking for transport and daily access to food '
    'without a car.  In Mexicali, where car ownership is '
    'high and summer heat suppresses discretionary '
    'walking, short trip distances to everyday '
    'destinations are a precondition for any walking at '
    'all, and the households least able to substitute a '
    'car trip are those on the lowest incomes.'
)
example['rationale']['health_pathways'] = [
    'physical_activity_transport',
    'food_environment',
]
example['rationale']['arid_context'] = (
    'Distance thresholds calibrated in temperate cities '
    'likely overstate walking here: in summer, shade and '
    'time of day plausibly bind before distance does.  '
    'Results should be read alongside the thermal comfort '
    'indicators from WP02.'
)
example['rationale']['evidence'] = [
    {
        'claim': 'Greater neighbourhood destination access '
                 'is associated with more walking for '
                 'transport and higher total physical '
                 'activity.',
        'citation': 'TODO: replace with the systematic '
                    'review you select, e.g. a review of '
                    'built environment and walking for '
                    'transport',
        'doi': None,
        'url': None,
        'evidence_type': 'systematic_review',
        'population': 'TODO',
        'exposure': 'Destination accessibility',
        'outcome': 'Walking for transport',
        'effect': 'TODO: effect size with 95% CI',
        'threshold_support': 'TODO: what distance the '
                             'evidence supports',
    },
]
example['data_sources'] = [
    {
        'name': 'OpenStreetMap (via GHSCI Mexicali study '
                'region)',
        'custodian': 'OpenStreetMap contributors',
        'citation': 'OpenStreetMap contributors (2026). '
                    'Geofabrik Mexico extract, 10 April '
                    '2026.',
        'url': 'https://download.geofabrik.de/'
               'north-america/mexico.html',
        'date_retrieved': '2026-04-10',
        'licence': 'ODbL-1.0',
        'licence_url': 'https://opendatacommons.org/'
                       'licenses/odbl/',
        'redistributable': True,
        'spatial_resolution': 'vector points',
        'temporal_coverage': '2026',
        'condesa_coverage': 'full',
        'notes': 'Volunteered data; completeness varies '
                 'and is likely lower in newly developed '
                 'areas.',
    },
]
example['method']['summary'] = (
    'Convenience destination points were extracted from '
    'the GHSCI Mexicali destinations layer and counted '
    'within each 100 m grid cell, then expressed per '
    'square kilometre.  Cell values were aggregated to '
    'coarser reporting geographies as a population '
    'weighted mean, falling back to area weighting where '
    'no resident population is recorded.'
)
example['method']['notebook'] = (
    'notebooks/00_overview_and_schema.ipynb')
example['method']['condesa_treatment'] = (
    'Computed natively on the 100 m grid, which covers '
    'the full Condesa extent; OpenStreetMap coverage of '
    'the new development is, however, likely incomplete.'
)

uli.todos(example)

In [ ]:
# 2. Calculation at the native scale.
convenience = destinations[
    destinations['dest_name'] == 'convenience']

native = uli.count_features(
    convenience, 'grid_100m', per='sqkm')
native['value'].describe()

In [ ]:
# 3. Harmonise to every reporting geography, label, check.
example['measures'] = [
    m for m in example['measures'] if m['lens'] == 'density'
]
measure = example['measures'][0]
measure['id'] = 'access_to_minimarts__density_per_sqkm'
measure['name_en'] = 'Convenience destinations per km²'
measure['description'] = (
    'Count of OpenStreetMap convenience destinations whose '
    'point falls within the unit, divided by unit area in '
    'km².')
measure['unit'] = 'count per km2'
measure['value_type'] = 'rate'
measure['direction'] = 'higher_is_better'
measure['denominator_type'] = 'area_sqkm'
measure['native_scale'] = 'grid_100m'
measure['aggregation_method'] = 'population_weighted_mean'

harmonised = uli.harmonise(
    native, 'grid_100m', method='population_weighted_mean')
results = uli.label(
    harmonised, example, measure['id'])
results.head()

In [ ]:
print(uli.check(results, example))

The report above will still fail: the `TODO` placeholders
in the evidence block are exactly the work this project
asks each analyst to do. That is the intended behaviour —
an indicator without independent health evidence is not
deliverable.

Once the evidence is real, `uli.write_indicator(results,
example)` writes the three deliverable files and the run is
reproducible from this notebook alone.

In [ ]:
results.groupby(['geo_level', 'aggregation_method']).agg(
    units=('value', 'size'),
    with_value=('value', 'count'),
    median=('value', 'median'),
)

Note the `condesa_fraccionamiento` rows: because the
calculation was native to the 100 m grid, all 40 units get
a value. Had it been native to `manzana`, only 33 would —
the seven fraccionamientos with no overlapping census block
would have been silently absent. Note too the
`area_weighted_mean` rows: those are units where the
population weight was zero and the fallback took over.

---
## Ingestion

The composite index and Reimagina Urbana both read
deliverables through the same function you can run
yourself:

In [ ]:
delivered, catalogue = uli.collect()
print(f'{len(delivered):,} rows from '
      f'{catalogue["indicator_code"].nunique() if len(catalogue) else 0} '
      'indicators')
catalogue.head(20) if len(catalogue) else 'Nothing delivered yet.'

In [ ]:
# Wide format for one geography, and a geopackage for QGIS
# / the platform:
# wide = uli.to_wide(delivered, 'manzana')
# uli.to_geopackage(delivered, 'outputs/mexicali_uli.gpkg')